### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
    * [1.2 Dataset loading/inspection](#12-dataset-loadinginspection)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 PSD](#31-psd-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)
* [5. Model Training](#5-model-training)
    * [Emotional vs. Neutral](#51-emotional-vs-neutral)
        * [XGBoost](#511-xgboost)




### 1. Environment Setup

##### 1.1 Library Imports

In [85]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import welch

import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, StratifiedGroupKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, classification_report


##### 1.2 Dataset loading/inspection

In [77]:
dataset_path = Path('DEED')
eeg_dataset = []
subject_ids = []

for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    subject_part = [part for part in fname.split("_") if part.startswith("S")][0]
    label = int(label_part[1:])  
    subject_id = subject_part[1:-1]  # last two digits = subject number
    
    eeg_dataset.append((eeg, label))
    subject_ids.append(subject_id)

print(f"Loaded {len(eeg_dataset)} trials.")
print(f"Unique subjects: {len(set(subject_ids))}")
print(f"Subject IDs: {sorted(set(subject_ids))}")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

Loaded 533 trials.
Unique subjects: 34
Subject IDs: ['002', '003', '004', '005', '007', '011', '012', '013', '014', '015', '016', '017', '018', '020', '021', '022', '023', '024', '025', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '042']
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]

Sample filename → label mapping:
  G_S0321_M1_E2_R1_N2_raw_ref → E2
  G_S0213_M3_E2_R7_N2_raw_ref → E2
  G_S0393_M2_E3_R5_REM_raw_ref → E3
  G_S0243_M3_E2_R2_N2_raw_ref → E2
  G_S0311_M3_E5_R2_N2_raw_ref → E5
  G_S0031_M1_E3_R4_nan_raw_ref → E3
  G_S0043_M2_E2_R5_N2_raw_ref → E2
  G_S0072_M1_E0_R11_N1_raw_ref → E0
  G_S0242_M1_E3_R3_W_raw_ref → E3
  G_S0342_M2_E3_R3_N2_raw_ref → E3


### 2. Data Segmentation

In [78]:
#Segment into 20s windows
def segmentation(eeg_dataset, subject_ids, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []
    groups = []

    for (eeg_array, label), sid in zip(eeg_dataset, subject_ids):
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            groups.append(sid)
            start += window_size  

    return X, y, groups

windows, window_labels, window_groups = segmentation(eeg_dataset, subject_ids, 20, 200)
print(f"Total 20 second windows: {len(windows)}")
print(f"Unique subjects: {len(set(window_groups))}")

Total 20 second windows: 7490
Unique subjects: 34


### 3. Feature Extraction & Class Distribution

##### 3.1 PSD Feature Extraction

In [79]:
def extract_psd_features(segmented_windows, labels, fs=200):
    freq_bands = {
        'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30),
            'gamma': (30, 45)
    }
    
    windows = np.array(segmented_windows)
    n_windows, n_channels, n_samples = windows.shape
    
    nperseg = min(512, n_samples)  
    noverlap = nperseg // 2
    
    all_features = []
    
    for ch_idx in range(n_channels):
        ch_data = windows[:, ch_idx, :]
        f, Pxx = welch(ch_data, fs=fs, nperseg=nperseg, noverlap=noverlap, axis=1)
        
        # Band power per band
        ch_band_powers = []
        for band_name, (low, high) in freq_bands.items():
            idx = np.logical_and(f >= low, f <= high)
            band_power = np.trapz(Pxx[:, idx], f[idx], axis=1) 
            ch_band_powers.append(band_power)
        ch_band_powers = np.column_stack(ch_band_powers) 

        # Log band power
        log_band_power = np.log(ch_band_powers + 1e-10)
        all_features.append(log_band_power)

        # Relative band power
        total_power = ch_band_powers.sum(axis=1, keepdims=True)
        relative_band_power = ch_band_powers / (total_power + 1e-10)
        all_features.append(relative_band_power)

        # Time-domain features
        ch_mean = np.mean(ch_data, axis=1, keepdims=True)
        ch_var = np.var(ch_data, axis=1, keepdims=True)
        all_features.append(ch_mean)
        all_features.append(ch_var)

        
    
    
    X = np.hstack(all_features)
    y = np.array(labels)
    
    return X, y

##### 3.2 Class Distribution

In [80]:
X, y = extract_psd_features(windows, window_labels)
print(X.shape)
print(y.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y)*100:.1f}%)")

(7490, 72)
(7490,)

=== Total Class Distribution===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [81]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_en, mask_en = remap_emotional_neutral(y)
X_en = X[mask_en]
groups_en = np.array(window_groups)[mask_en]

print(f"\n=== Emotional vs. Neutral Class Distribution ===")
print(f"  Total: {len(y_en)}")
print(f"  Neutral (0): {np.sum(y_en == 0)} ({np.sum(y_en == 0)/len(y_en)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y_en == 1)} ({np.sum(y_en == 1)/len(y_en)*100:.1f}%)")


=== Emotional vs. Neutral Class Distribution ===
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [82]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_pn, mask_pn = remap_positive_negative(y)
X_pn = X[mask_pn]
groups_pn = np.array(window_groups)[mask_pn]

print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y_pn)}")
print(f"  Negative (0): {np.sum(y_pn == 0)} ({np.sum(y_pn == 0)/len(y_pn)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y_pn == 1)} ({np.sum(y_pn == 1)/len(y_pn)*100:.1f}%)")


=== Positive vs. Negative Class Distribution ===
  Total: 3341
  Negative (0): 1374 (41.1%)
  Positive (1): 1967 (58.9%)


### 5. Model Training
For both models and classification schemes, hyperparameters are tuned once using Optuna with StratifiedGroupKFold (5 splits) on the full dataset, ensuring no subject appears in both train and validation folds. Best parameters are then frozen and used for final LOSO evaluation.

##### 5.1 Emotional vs. Neutral

##### 5.1.1 XGBoost

In [ ]:
def no_improvement_callback(study, trial, n_trials_no_improve=20):
    if trial.number >= n_trials_no_improve:
        recent_values = [t.value for t in study.trials[-n_trials_no_improve:]]
        if max(recent_values) <= study.best_value:
            study.stop()

# Class balancing
neg_count = np.sum(y_en == 0)
pos_count = np.sum(y_en == 1)
scale_en = neg_count / pos_count

# Tune once on full dataset with StratifiedGroupKFold
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 300),
        'max_depth':        trial.suggest_int('max_depth', 3, 6),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': scale_en,
        'eval_metric':      'logloss',
    }
    model = XGBClassifier(**params, n_jobs=-1)
    cv = StratifiedGroupKFold(n_splits=5)
    scores = cross_val_score(model, X_en, y_en, cv=cv, groups=groups_en, scoring='accuracy', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, callbacks=[lambda s, t: no_improvement_callback(s, t)])
print(f"Best params: {study.best_params}")
print(f"Best CV accuracy: {study.best_value:.4f}")

# Freeze best params
best_params = study.best_params
best_params['scale_pos_weight'] = scale_en
best_params['random_state'] = 42

# LOSO evaluation with fixed params
logo = LeaveOneGroupOut()
en_accuracies = []
en_f1s = []
en_aucs = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X_en, y_en, groups_en)):
    X_train, X_test = X_en[train_idx], X_en[test_idx]
    y_train, y_test = y_en[train_idx], y_en[test_idx]

    model = XGBClassifier(**best_params, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    en_accuracies.append(accuracy_score(y_test, preds))
    en_f1s.append(f1_score(y_test, preds, average='macro'))
    en_aucs.append(roc_auc_score(y_test, proba))
    print(f"Fold {fold+1} | Accuracy: {en_accuracies[-1]:.4f} | F1: {en_f1s[-1]:.4f} | AUC: {en_aucs[-1]:.4f}")

print("\n=== XGBoost — Emotional vs Neutral ===")
print(f"Accuracy:  {np.mean(en_accuracies):.4f} ± {np.std(en_accuracies):.4f}")
print(f"F1 Macro:  {np.mean(en_f1s):.4f} ± {np.std(en_f1s):.4f}")
print(f"ROC-AUC:   {np.mean(en_aucs):.4f} ± {np.std(en_aucs):.4f}")